In [1]:
import os
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np

In [3]:
from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from sklearn.pipeline import make_pipeline

data_path = Path('data')

loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    PandasConverter(index_col = 'id')
)
df_train = loader.fit_transform([data_path / 'train.csv'])
df_test = loader.transform([data_path / 'test.csv'])
df_org = loader.transform([data_path / 'irrigation_prediction.csv'])
df_org = df_org.set_index(pd.Index(-np.arange(1, len(df_org) + 1), name = 'id'))

y = 'Irrigation_Need'
X_cat = ['Crop_Growth_Stage', 'Crop_Type', 'Irrigation_Type', 'Mulching_Used', 'Region', 'Season', 'Soil_Type', 'Water_Source']
X_num = ['Electrical_Conductivity', 'Field_Area_hectare', 'Humidity', 'Organic_Carbon', 'Previous_Irrigation_mm', 'Rainfall_mm', 
         'Soil_Moisture', 'Soil_pH', 'Sunlight_Hours', 'Temperature_C','Wind_Speed_kmh']
X_all = X_num + X_cat

In [17]:
from mllabs import Experimenter
import lightgbm as lgb

## Train과 Test의 분포 차이 확인

In [11]:
df_diff = pd.concat([
    df_train.drop(columns=y).assign(is_train = True),
    df_test.assign(is_train = False),
])

In [12]:
e_diff = Experimenter.create(df_diff, 'exp/diff')

📁 Created directory: exp/diff


In [29]:
e_diff.set_grp('clf', method = 'predict_proba', role = 'head', edges = {'y': [(None, 'is_train')]})

Group 'clf' updated, 1 node(s) affected


{'result': 'update',
 'affected_nodes': ['lgb'],
 'old_grp': <mllabs._pipeline.PipelineGroup at 0x7fa96bfba240>,
 'grp': <mllabs._pipeline.PipelineGroup at 0x7fa940fcb530>}

In [30]:
from mllabs.collector import MetricCollector
from mllabs import Connector
from sklearn.metrics import roc_auc_score
e_diff.add_collector(
    MetricCollector(
        'AUC', Connector(edges = {'y': [(None, 'is_train')]}, role = 'head'), slice(-1, None), roc_auc_score, include_train = True
    )
)

In [32]:
e_diff.set_node('lgb', grp='clf', processor=lgb.LGBMClassifier, edges={'X': [(None, X_all)]}, params={'verbose': -1})
e_diff.exp()

Experimenting 1 node(s)
Exp complete: 1 node(s)


In [34]:
e_diff.get_collector('AUC').get_metrics_agg()[0]

,test,train
lgb,0.498932,0.576174


In [8]:
df_train[X_all].value_counts().max()

np.int64(1)

- 중복되는 입력값은 없음